# Clubfoot Condition Classifier
### Fine-tuning a vision model on your labeled training data

**What this notebook does:**
1. Loads your exported training data from the Clubfoot Club app
2. Shows you the class distribution (you'll see the imbalance immediately)
3. Fine-tunes a pretrained ResNet-18 on your images
4. Plots training vs validation loss (this is where you *see* overfitting)
5. Generates a confusion matrix so you know exactly which conditions get confused

**Before you start:** In Colab, go to `Runtime → Change runtime type → T4 GPU`. Training is ~10x faster on GPU.

**To get your data:** In the app, open the `/train` page and click Export. Save the JSON file. You'll upload it in Step 2.

---
## Step 1 — Install dependencies and imports

In [ ]:
# PyTorch and torchvision are pre-installed in Colab.
# We only need scikit-learn for the train/val/test split and confusion matrix.
!pip install -q scikit-learn

import json, base64, io, os, random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("  ⚠ No GPU found. Training will work but will be slower. Consider Runtime → Change runtime type → T4 GPU")

---
## Step 2 — Load your exported data

Run the cell below, then click the **Choose Files** button and select the JSON file you exported from the app.

In [ ]:
from google.colab import files

print("Upload your exported JSON file from the Clubfoot Club /train page:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
raw_records = json.loads(uploaded[filename])
print(f"\nLoaded {len(raw_records)} total records from '{filename}'")

### What a record looks like

Each record has a `correction` field (the ground-truth label a human assigned) and an `imageData` field (the photo as a base64 string). That's the entire training signal.

In [ ]:
# Peek at one record (without printing the full base64 blob)
sample = {k: v for k, v in raw_records[0].items() if k != "imageData"}
sample["imageData"] = "<base64 image string, omitted for readability>"
print(json.dumps(sample, indent=2))

---
## Step 3 — Filter and clean the records

We keep only records that have:
- A valid `correction` label (the human-assigned ground truth)
- Actual image data
- A non-meta condition (we skip `image_unclear` and `no_relevant_anatomy` — those are about photo quality, not medical conditions)

We also exclude conditions that only appear in the meta domain.

In [ ]:
# These are your condition ids from conditions.js — the full label set.
# Meta conditions are excluded: they describe image quality, not clinical findings.
CLINICAL_CONDITIONS = [
    "cast_normal",
    "cast_too_tight",
    "cast_wet_or_damaged",
    "cast_loose",
    "brace_normal",
    "brace_heel_not_seated",
    "brace_blister_or_redness",
    "brace_bar_issue",
    "foot_normal_position",
    "foot_relapse_signs",
    "foot_toe_walking",
]

# Build label → integer index mapping
LABEL_TO_IDX = {label: i for i, label in enumerate(CLINICAL_CONDITIONS)}
IDX_TO_LABEL = {i: label for label, i in LABEL_TO_IDX.items()}

# Filter records
valid_records = [
    r for r in raw_records
    if r.get("correction") in LABEL_TO_IDX
    and r.get("imageData")
]

print(f"Total records:   {len(raw_records)}")
print(f"Usable records:  {len(valid_records)}")
print(f"Dropped:         {len(raw_records) - len(valid_records)} (no label, meta condition, or missing image)")

---
## Step 4 — Visualize class distribution

This is the first thing you should always look at. **Imbalanced classes are the number one thing that silently kills a classifier.** A model trained on 500 cast_normal and 8 brace_bar_issue examples will learn to basically never predict brace_bar_issue — and it will still look like it has 95% accuracy.

In [ ]:
label_counts = Counter(r["correction"] for r in valid_records)

# Sort by count descending for readability
sorted_items = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)
labels_sorted, counts_sorted = zip(*sorted_items)

colors = ["#ef4444" if c < 50 else "#f59e0b" if c < 100 else "#22c55e" for c in counts_sorted]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(labels_sorted[::-1], counts_sorted[::-1], color=colors[::-1])
ax.set_xlabel("Number of labeled examples")
ax.set_title("Training data distribution by condition")

# Add count labels
for bar, count in zip(bars, counts_sorted[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(color="#ef4444", label="< 50 examples (danger zone)"),
    Patch(color="#f59e0b", label="50–99 examples (borderline)"),
    Patch(color="#22c55e", label="≥ 100 examples (workable)"),
]
ax.legend(handles=legend, loc="lower right")
plt.tight_layout()
plt.show()

print("\nFull counts:")
for label, count in sorted_items:
    bar = "█" * min(count, 50)
    flag = " ← needs more data" if count < 50 else ""
    print(f"  {label:<30} {count:>4}  {bar}{flag}")

---
## Step 5 — Build the PyTorch Dataset

PyTorch's `Dataset` class is how you tell the training loop "here's how to load one example." It needs two methods:
- `__len__`: how many examples total
- `__getitem__(i)`: give me example number i (returns image tensor + label integer)

We also define **data augmentation** here — random flips, color jitter, etc. This synthetically multiplies your dataset and forces the model to generalize rather than memorize specific photos.

In [ ]:
def decode_image(base64_data_url):
    """Convert a base64 data URL (from the app) to a PIL Image."""
    # Strip the data:image/jpeg;base64, prefix
    if "," in base64_data_url:
        base64_data_url = base64_data_url.split(",", 1)[1]
    image_bytes = base64.b64decode(base64_data_url)
    return Image.open(io.BytesIO(image_bytes)).convert("RGB")


# ImageNet normalization — ResNet was pretrained on ImageNet, so we normalize
# our images to the same distribution it expects. Without this, the pretrained
# weights are essentially calibrated to the wrong input range.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transform: resize + augment + normalize
# Augmentation = random variations that preserve the label but change the pixels.
# The model sees a different version of each photo on every epoch, which forces
# it to learn features that survive flipping, brightness changes, etc.
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),           # randomly crop to 224×224
    transforms.RandomHorizontalFlip(),    # flip left-right randomly
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3,     # vary brightness/contrast
        saturation=0.2, hue=0.05,
    ),
    transforms.RandomRotation(10),        # slight tilt
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Validation/test transform: deterministic — no augmentation
# We want a stable measurement of accuracy, not a random one.
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ClubfootDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image = decode_image(record["imageData"])
        image_tensor = self.transform(image)
        label_idx = LABEL_TO_IDX[record["correction"]]
        return image_tensor, label_idx


print("Dataset class defined.")
print(f"Number of classes: {len(CLINICAL_CONDITIONS)}")
print(f"Classes: {CLINICAL_CONDITIONS}")

---
## Step 6 — Split into train / validation / test sets

**This split is sacred.** The rule:
- **Train** (70%): the model sees these images and learns from them
- **Validation** (15%): used *during training* to measure if you're overfitting (never trained on)
- **Test** (15%): locked in a vault; only opened at the very end to get your final honest accuracy number

We use `stratify=labels` to ensure each split has the same class proportions — otherwise you might get a test set with zero examples of your rarest class.

In [ ]:
all_labels = [r["correction"] for r in valid_records]

# Check if we have enough data to split
min_count = min(Counter(all_labels).values())
if min_count < 3:
    print(f"⚠  Some classes have fewer than 3 examples. Stratified split will fail.")
    print(f"   Classes with < 3 examples:")
    for label, count in Counter(all_labels).items():
        if count < 3:
            print(f"   - {label}: {count}")
    print("\n   You need at least 3 examples per class (1 for each split).")
    print("   Collect more data for these classes before training.")
    raise SystemExit("Not enough data to split.")

# First split: 85% temp, 15% test
train_val_records, test_records = train_test_split(
    valid_records, test_size=0.15, random_state=42,
    stratify=[r["correction"] for r in valid_records]
)

# Second split: 70% train, 15% validation (of the original 85%)
train_records, val_records = train_test_split(
    train_val_records, test_size=0.176, random_state=42,  # 0.176 * 0.85 ≈ 0.15
    stratify=[r["correction"] for r in train_val_records]
)

print(f"Train:      {len(train_records):>4} records ({100*len(train_records)/len(valid_records):.0f}%)")
print(f"Validation: {len(val_records):>4} records ({100*len(val_records)/len(valid_records):.0f}%)")
print(f"Test:       {len(test_records):>4} records ({100*len(test_records)/len(valid_records):.0f}%)")

### Handle class imbalance with a weighted sampler

If `cast_normal` has 400 examples and `brace_bar_issue` has 8, a naive training loop sees `cast_normal` 50× more often. The model learns to predict `cast_normal` for everything and still looks 80% accurate. 

**WeightedRandomSampler** fixes this by oversampling rare classes so each class appears roughly equally often during training — the model has to get them all right.

In [ ]:
# Compute per-class weights: weight = 1 / count
# Rare classes get high weight (sampled more), common classes get low weight.
train_label_counts = Counter(r["correction"] for r in train_records)
class_weights = {label: 1.0 / count for label, count in train_label_counts.items()}

# Assign each training record its weight
sample_weights = [class_weights[r["correction"]] for r in train_records]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_records),
    replacement=True,  # allows oversampling rare classes
)

# Build datasets and DataLoaders
train_dataset = ClubfootDataset(train_records, train_transform)
val_dataset   = ClubfootDataset(val_records,   eval_transform)
test_dataset  = ClubfootDataset(test_records,  eval_transform)

BATCH_SIZE = 32  # How many images to process before updating weights

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("DataLoaders ready.")
print(f"Batches per epoch: {len(train_loader)} (batch size {BATCH_SIZE})")

---
## Step 7 — Build the model

We use **ResNet-18** — a convolutional neural network with 18 layers that was pretrained on ImageNet (1.2 million images, 1000 classes). It already knows what edges, textures, shapes, and objects look like.

**Transfer learning strategy:**
1. Take the whole pretrained ResNet-18
2. Replace its final classification layer (which predicted 1000 ImageNet classes) with a new layer that predicts your 11 conditions
3. Freeze the early layers (let them keep their ImageNet knowledge untouched)
4. Only train the last few layers + the new head on your data

This means you're not teaching the model to see from scratch — you're just teaching it to recognize *your specific patterns* using general visual features it already knows.

In [ ]:
NUM_CLASSES = len(CLINICAL_CONDITIONS)

# Load ResNet-18 with ImageNet pretrained weights
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# --- FREEZE early layers ---
# The early layers detect generic features (edges, corners, textures).
# Freezing them = setting requires_grad=False, which means gradient descent
# will NOT update those weights. They stay exactly as ImageNet left them.
# We only want to retrain the layers that need to learn clubfoot-specific patterns.
for name, param in model.named_parameters():
    # Freeze everything except layer4 and fc (the last residual block + classifier)
    if not (name.startswith("layer4") or name.startswith("fc")):
        param.requires_grad = False

# Replace the final fully-connected layer.
# ResNet-18's original fc layer: 512 → 1000 (ImageNet classes)
# Our new fc layer:              512 → NUM_CLASSES (our conditions)
in_features = model.fc.in_features  # = 512 for ResNet-18
model.fc = nn.Sequential(
    nn.Dropout(0.3),            # randomly zero out 30% of neurons during training
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)

# Count trainable vs frozen parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  ({100*trainable_params/total_params:.1f}% of the network)")
print(f"Frozen parameters:    {total_params - trainable_params:,}  (early layers, untouched)")

---
## Step 8 — Define the loss function and optimizer

**Loss function** measures how wrong the model is. We use **cross-entropy loss**, which is standard for multi-class classification. It penalizes confident wrong answers much harder than uncertain wrong answers.

**Optimizer** implements gradient descent. We use **Adam**, which adapts the learning rate for each weight individually — in practice it converges faster and more reliably than vanilla SGD for fine-tuning tasks.

**Learning rate scheduler** reduces the learning rate when validation loss stops improving. The model starts with bigger steps and takes smaller steps as it gets closer to a good solution.

In [ ]:
# Only pass parameters that require gradients to the optimizer.
# Passing frozen parameters would waste memory and confuse the optimizer.
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,       # learning rate — how big a step to take each update
    weight_decay=1e-4  # L2 regularization — gently penalizes very large weights
)

# Cross-entropy loss function
criterion = nn.CrossEntropyLoss()

# Reduce learning rate by 50% when validation loss hasn't improved for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

print("Loss function: CrossEntropyLoss")
print("Optimizer:     Adam (lr=1e-3)")
print("Scheduler:     ReduceLROnPlateau (patience=3, factor=0.5)")

---
## Step 9 — Training loop

This is where machine learning actually happens. Each **epoch** is one full pass through the training set.

```
For each batch of images:
  1. Forward pass  → model produces predictions
  2. Compute loss  → how wrong are the predictions?
  3. Backward pass → compute gradients (which direction reduces loss?)
  4. Optimizer step → nudge weights in that direction
  5. Zero gradients → reset for the next batch
```

Watch the **training loss** vs **validation loss** as they print. When training loss keeps falling but validation loss starts rising — that's overfitting. The model is memorizing training photos rather than learning general patterns.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, is_train=True):
    """Run one epoch. If optimizer is provided, updates weights (training mode)."""
    model.train(is_train)
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # --- FORWARD PASS ---
        # Disable gradient computation during validation (saves memory + time)
        with torch.set_grad_enabled(is_train):
            logits = model(images)          # raw scores, one per class
            loss = criterion(logits, labels)  # cross-entropy loss

        if is_train and optimizer:
            # --- BACKWARD PASS ---
            optimizer.zero_grad()  # clear gradients from previous batch
            loss.backward()        # compute new gradients (backpropagation)
            optimizer.step()       # update all weights by one step

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


NUM_EPOCHS = 30  # Increase if you have a lot of data and want to train longer

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0
EARLY_STOP_PATIENCE = 8

print(f"Training for up to {NUM_EPOCHS} epochs (early stop after {EARLY_STOP_PATIENCE} epochs without improvement)")
print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  {'Train Acc':>10}  {'Val Acc':>10}  {'LR':>8}")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, is_train=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion,             is_train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    improved = "✓" if val_loss < best_val_loss else " "
    print(f"{epoch:>6}  {train_loss:>10.4f}  {val_loss:>10.4f}  {train_acc:>9.1%}  {val_acc:>9.1%}  {current_lr:>8.2e}  {improved}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

print(f"\nBest validation loss: {best_val_loss:.4f}")

# Restore best weights
model.load_state_dict(best_model_state)
print("Restored best model weights.")

---
## Step 10 — Plot the loss curves

This chart is the most important visual in ML training. Learn to read it:

- **Both curves falling together**: healthy training
- **Val loss flattens while train loss keeps falling**: overfitting — the model has memorized training data
- **Val loss is lower than train loss**: suspicious — usually means the model isn't actually learning much (or training data augmentation is very aggressive)
- **Both curves flat from epoch 1**: learning rate is too low, or frozen too many layers

In [ ]:
epochs_run = len(history["train_loss"])
epoch_range = range(1, epochs_run + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss curves
ax1.plot(epoch_range, history["train_loss"], label="Train loss",      color="#3b82f6", linewidth=2)
ax1.plot(epoch_range, history["val_loss"],   label="Validation loss",  color="#ef4444", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training vs Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Mark where best val loss occurred
best_epoch = history["val_loss"].index(min(history["val_loss"])) + 1
ax1.axvline(x=best_epoch, color="#22c55e", linestyle="--", alpha=0.7, label=f"Best epoch ({best_epoch})")
ax1.legend()

# Accuracy curves
ax2.plot(epoch_range, history["train_acc"], label="Train accuracy",      color="#3b82f6", linewidth=2)
ax2.plot(epoch_range, history["val_acc"],   label="Validation accuracy",  color="#ef4444", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training vs Validation Accuracy")
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Overfitting diagnosis
gap = history["train_acc"][-1] - history["val_acc"][-1]
print(f"\nFinal train accuracy:      {history['train_acc'][-1]:.1%}")
print(f"Final validation accuracy: {history['val_acc'][-1]:.1%}")
print(f"Generalization gap:        {gap:.1%}")
if gap > 0.15:
    print("\n⚠  Large gap — model is overfitting. To fix:")
    print("   • Collect more labeled data (biggest lever)")
    print("   • Increase Dropout (e.g. 0.3 → 0.5) in model.fc")
    print("   • Increase data augmentation")
    print("   • Freeze more layers (freeze layer3 as well)")
elif gap < 0.02:
    print("\nℹ  Very small gap — model might be underfitting. Try:")
    print("   • Unfreeze layer3 (more capacity)")
    print("   • Train more epochs")
    print("   • Increase learning rate")
else:
    print("\n✓  Healthy generalization gap.")

---
## Step 11 — Evaluate on the test set

This is the first and only time we touch the test set. The number we get here is our **honest estimate of real-world performance** — it's not contaminated by any training decisions (unlike validation accuracy, which we implicitly optimized by stopping when val loss was lowest).

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

test_accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test set accuracy: {test_accuracy:.1%}  (on {len(all_labels)} held-out examples)")
print("\nPer-class report:")
print(classification_report(
    all_labels, all_preds,
    target_names=CLINICAL_CONDITIONS,
    zero_division=0
))

---
## Step 12 — Confusion matrix

**Accuracy is a lie for imbalanced datasets.** The confusion matrix is what you actually want. Each cell (row i, column j) shows: of all images that were actually class i, how many did the model predict as class j? The diagonal = correct predictions. Off-diagonal = mistakes.

The medically important question: does the model confuse urgent conditions with normal ones?

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

# Normalize by row so each cell shows % of that true class
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)  # handle classes with 0 test examples

# Short names for readability
short_names = [
    c.replace("cast_", "c_")
     .replace("brace_", "b_")
     .replace("foot_", "f_")
     .replace("_or_", "/")
     .replace("_not_", "!")
    for c in CLINICAL_CONDITIONS
]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_normalized, annot=True, fmt=".0%", cmap="Blues",
    xticklabels=short_names, yticklabels=short_names,
    ax=ax, linewidths=0.5, vmin=0, vmax=1
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title("Confusion Matrix (row = actual, col = predicted, normalized)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Flag dangerous confusions: urgent predicted as normal
URGENT_CLASSES   = ["cast_too_tight", "cast_wet_or_damaged", "brace_bar_issue", "foot_relapse_signs"]
NORMAL_CLASSES   = ["cast_normal", "brace_normal", "foot_normal_position"]
urgent_idx  = [LABEL_TO_IDX[c] for c in URGENT_CLASSES  if c in LABEL_TO_IDX]
normal_idx  = [LABEL_TO_IDX[c] for c in NORMAL_CLASSES  if c in LABEL_TO_IDX]

print("\nDangerous confusions (urgent predicted as normal):")
found = False
for ui in urgent_idx:
    for ni in normal_idx:
        if cm[ui][ni] > 0:
            print(f"  ⚠  {CLINICAL_CONDITIONS[ui]} → {CLINICAL_CONDITIONS[ni]}: {cm[ui][ni]} cases")
            found = True
if not found:
    print("  ✓ None found.")

---
## Step 13 — Save the model

Save the model weights and configuration so you can reload it for inference later.

In [ ]:
save_path = "clubfoot_classifier.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "label_to_idx": LABEL_TO_IDX,
    "idx_to_label": IDX_TO_LABEL,
    "conditions": CLINICAL_CONDITIONS,
    "architecture": "resnet18",
    "test_accuracy": test_accuracy,
}, save_path)

print(f"Model saved to {save_path}")

# Download it
from google.colab import files
files.download(save_path)
print("Download started.")

---
## Step 14 — Run inference on a new image

This is how you'd use the trained model on a fresh photo — the same pipeline the app would call if you swapped out the Claude API.

In [ ]:
import torch.nn.functional as F

def predict(model, image_source, top_k=3):
    """
    image_source: a PIL Image, a file path string, or a base64 data URL
    Returns: list of (condition_id, probability) tuples, sorted by probability desc
    """
    if isinstance(image_source, str):
        if image_source.startswith("data:"):
            image = decode_image(image_source)
        else:
            image = Image.open(image_source).convert("RGB")
    else:
        image = image_source.convert("RGB")

    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)  # add batch dimension

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze().cpu().tolist()

    results = sorted(
        [(IDX_TO_LABEL[i], probs[i]) for i in range(len(probs))],
        key=lambda x: x[1], reverse=True
    )
    return results[:top_k]


# Test it on a random image from the test set
sample_record = random.choice(test_records)
sample_image  = decode_image(sample_record["imageData"])
true_label    = sample_record["correction"]

predictions = predict(model, sample_image)

# Display the image
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(sample_image)
ax.set_title(f"True label: {true_label}", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nTrue label: {true_label}")
print("Top predictions:")
for condition, prob in predictions:
    bar = "█" * int(prob * 30)
    marker = " ← correct" if condition == true_label else ""
    print(f"  {condition:<35} {prob:>5.1%}  {bar}{marker}")

---
## Where to go from here

### If your model is overfitting (train acc >> val acc)
- **Biggest lever:** collect more labeled images for the rare classes
- Increase Dropout from 0.3 → 0.5 in `model.fc`
- Add more aggressive augmentation (random erasing, mixup)
- Freeze more layers — try freezing `layer3` as well

### If your model is underfitting (low accuracy on both)
- Unfreeze `layer3` to give the model more capacity
- Increase learning rate slightly
- Train more epochs (raise `NUM_EPOCHS`)
- Try a bigger backbone: replace ResNet-18 with ResNet-50 (`models.resnet50`)

### How to get more training data
1. **Fix the PDF importer** — render at 2× scale and increase `max_tokens` to 2048 in `pdfImport.js`
2. **The `/train` page** — every parent who uses the scan tool and marks "this was wrong" generates a labeled example
3. **Direct input** — find Facebook posts with expert comments, screenshot the photo, label it on `/train`

### How this connects back to the app
Right now the app calls Claude (a 70B+ parameter LLM) for every scan. That's powerful but slow (~3s) and costs API credits. Once you have a well-performing fine-tuned classifier, you could:
- Run it client-side via **ONNX** or **TensorFlow.js** (zero API cost, instant)
- Use it as a **pre-filter** before Claude — only escalate to Claude when confidence is low
- Use Claude to handle the *reasoning and careTeamMessage* while your classifier handles the fast *condition + urgency* judgment

### Key ML concepts you just used
| Concept | Where you saw it |
|---|---|
| Forward pass | `logits = model(images)` |
| Loss function | `criterion(logits, labels)` |
| Backpropagation | `loss.backward()` |
| Gradient descent step | `optimizer.step()` |
| Frozen parameters | `param.requires_grad = False` |
| Transfer learning | Pretrained ResNet-18 + new head |
| Overfitting | Train loss falling, val loss rising |
| Class imbalance | `WeightedRandomSampler` |
| Regularization | Dropout + weight_decay |
| Early stopping | `epochs_without_improvement` counter |